# Experiments Notebook

This notebook runs the evaluation and downstream classification experiments using the project's scripts. It is designed so you can run individual steps (inference, evaluation, epoch-based classifier) from within the notebook and inspect the outputs.


In [ ]:
# Imports and helpers
import os, sys, json, subprocess, shlex
from IPython.display import Image, display
import matplotlib.pyplot as plt

ROOT = os.getcwd()
TRAIN_DIR = 'data/processed/train'
TEST_GT_DIR = 'data/processed/test'
TEST_RECON_DIR = 'outputs/inference_test'
OUT_DIR = 'outputs/analysis'

def run_cmd(cmd, cwd=None):
    print('RUNNING:', cmd)
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    p = subprocess.run(cmd, cwd=cwd or ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return p.stdout

print('root:', ROOT)
print('train exists:', os.path.exists(TRAIN_DIR))
print('test_gt exists:', os.path.exists(TEST_GT_DIR))
print('test_recon exists:', os.path.exists(TEST_RECON_DIR))
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
# Controls: set to True to run the corresponding step from the notebook (be careful - inference can be slow).
RUN_INFER = False
RUN_EVAL = False
RUN_EPOCHS = False
INFER_LIMIT = 100  # pass --limit to infer.py for quicker runs
EPOCHS = 20

if RUN_INFER:
    cmd = f'python scripts/infer.py --src {TEST_GT_DIR} --out outputs/inference_test --ckpt model.pth --limit {INFER_LIMIT}'
    run_cmd(cmd)

if RUN_EVAL:
    # Example: run evaluation for each class folder found in outputs/inference_test
    for cls in sorted(os.listdir(TEST_RECON_DIR)):
        cls_folder = os.path.join(TEST_RECON_DIR, cls)
        if not os.path.isdir(cls_folder):
            continue
        out_folder = os.path.join('outputs','eval_test', cls)
        os.makedirs(out_folder, exist_ok=True)
        cmd = f'python evaluation/evaluate.py --pred-folder {cls_folder} --out {out_folder}'
        run_cmd(cmd)

if RUN_EPOCHS:
    cmd = f'python scripts/run_classification_epochs.py --train {TRAIN_DIR} --test-gt {TEST_GT_DIR} --test-recon {TEST_RECON_DIR} --out {OUT_DIR} --size 64 --epochs {EPOCHS}'
    run_cmd(cmd)

print('Controlled steps complete. Set RUN_* flags and re-run cells to execute.')


In [ ]:
# Load and plot training history if it exists
hist_path = os.path.join(OUT_DIR, 'training_history.json')
if os.path.exists(hist_path):
    with open(hist_path) as f:
        hist = json.load(f)
    epochs = hist.get('epoch', [])
    plt.figure(figsize=(8,4))
    for key in ['train_acc','test_gt_acc','test_recon_acc']:
        if key in hist:
            plt.plot(epochs, hist[key], label=key)
    plt.xlabel('epoch')
    plt.ylabel('accuracy')
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8,4))
    for key in ['train_loss','test_gt_loss','test_recon_loss']:
        if key in hist:
            plt.plot(epochs, hist[key], label=key)
    plt.xlabel('epoch')
    plt.ylabel('log_loss')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print('No training_history.json found at', hist_path)


In [ ]:
# Display confusion matrices and curves if present
for fn in ['confusion_gt_epochs.png','confusion_recon_epochs.png','accuracy_curve.png','loss_curve.png']:
    p = os.path.join(OUT_DIR, fn)
    if os.path.exists(p):
        print(p)
        display(Image(p))
    else:
        print('missing', p)


Notes:
- This notebook shells out to the existing scripts for reproducibility.
- Set RUN_* flags above to execute heavy steps from the notebook.
- If you want I can inline the model inference code here instead of calling scripts.